# Crop Recommendation System: Model Selection, Explanation, Prediction and Economic Ranking

**Version 6.** Adds a **soil analysis step** on top of v5 (cross-validated model selection, scaled SVM/MLP, all-class SHAP, seeded DiCE, corrected economics, pickle export). Main changes from v4:
- Best model is chosen by **5-fold cross-validation on the training set only** (accuracy, F1 as tie-breaker). The test set is used only for the final report.
- SVM and MLP are wrapped with `StandardScaler` for a fair comparison.
- SHAP shows **mean importance across all crop classes** (v4 plotted only class 0).
- DiCE is seeded and reports changes in **all 7 features**.
- Wrong per-quintal profit cells removed; only the corrected per-hectare economics remain.
- The best model is exported as a **pickle** bundle.
- **New in v6:** soil analysis compares your N, P, K and pH with the typical range of the recommended crop and suggests adjustments.

## 0. Setup

In [ ]:
!pip install -q lightgbm xgboost shap dice-ml

In [ ]:
import os, random, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

try:
    import lightgbm as lgb
except ImportError:
    lgb = None
    print("LightGBM not installed - it will be skipped.")
try:
    import xgboost as xgb
except ImportError:
    xgb = None
    print("XGBoost not installed - it will be skipped.")

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

def load_csv(name):
    """Load a CSV from the working folder, or ask for an upload in Colab."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        from google.colab import files
        print(f"Please upload '{name}':")
        up = files.upload()
        return pd.read_csv(list(up.keys())[0])
    except ImportError:
        raise FileNotFoundError(f"'{name}' not found in the working directory.")

## 1. Data Loading and Exploration

In [ ]:
df = load_csv("Crop_recommendation.csv")
print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
print("Missing values per column:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nNumber of crop classes:", df['label'].nunique())
display(df['label'].value_counts())

## 2. Preprocessing: Split and Label Encoding

In [ ]:
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Classes:", list(label_encoder.classes_))

### Typical range per crop (used later for soil analysis)
For each crop, the "ideal range" is the 10th to 90th percentile of that crop's samples in the dataset, and the typical value is the median.

In [ ]:
crop_profile = {
    crop: pd.DataFrame({
        'low':    g[FEATURES].quantile(0.10),
        'median': g[FEATURES].median(),
        'high':   g[FEATURES].quantile(0.90),
    }).T
    for crop, g in df.groupby('label')
}
display(crop_profile['rice'].round(2))

## 3. Define Candidate Models
SVM and MLP are scale-sensitive, so they are wrapped in a pipeline with `StandardScaler`. Tree models do not need scaling.

In [ ]:
models = {
    'SVM': make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)),
    'MLP': make_pipeline(StandardScaler(), MLPClassifier(max_iter=1000, random_state=SEED)),
    'Random Forest': RandomForestClassifier(random_state=SEED),
}
if lgb is not None:
    models['LightGBM'] = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
if xgb is not None:
    models['XGBoost'] = xgb.XGBClassifier(eval_metric='mlogloss', random_state=SEED)

# Stacking: base learners + logistic-regression meta-learner
base_learners = [
    ('svm', make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED))),
    ('mlp', make_pipeline(StandardScaler(), MLPClassifier(max_iter=1000, random_state=SEED))),
    ('rf', RandomForestClassifier(random_state=SEED)),
]
if lgb is not None:
    base_learners.append(('lgbm', lgb.LGBMClassifier(random_state=SEED, verbose=-1)))

models['Stacking'] = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=2000, random_state=SEED),
    cv=5, n_jobs=-1)

print("Candidate models:", list(models.keys()))

## 4. Model Selection by Cross-Validation (training data only)
The best model is picked by mean CV accuracy; mean CV weighted-F1 breaks ties. The test set is not touched here, so it stays an unbiased final check.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_rows = {}
for name, model in models.items():
    print(f"Cross-validating {name}...")
    res = cross_validate(model, X_train, y_train_encoded, cv=cv, n_jobs=-1,
                         scoring=['accuracy', 'f1_weighted'])
    cv_rows[name] = {
        'CV Accuracy': res['test_accuracy'].mean(),
        'CV Acc Std':  res['test_accuracy'].std(),
        'CV F1':       res['test_f1_weighted'].mean(),
    }

cv_df = (pd.DataFrame(cv_rows).T
         .sort_values(['CV Accuracy', 'CV F1'], ascending=False))
display(cv_df.round(4))

best_model_name = cv_df.index[0]
print(f"\nBest model by cross-validation: {best_model_name}")

## 5. Train All Models and Evaluate on the Held-out Test Set

In [ ]:
fitted = {}
test_rows = {}
for name, model in models.items():
    model.fit(X_train, y_train_encoded)
    fitted[name] = model
    pred = model.predict(X_test)
    test_rows[name] = {
        'Accuracy':  accuracy_score(y_test_encoded, pred),
        'Precision': precision_score(y_test_encoded, pred, average='weighted', zero_division=0),
        'Recall':    recall_score(y_test_encoded, pred, average='weighted', zero_division=0),
        'F1-Score':  f1_score(y_test_encoded, pred, average='weighted'),
    }

performance_df = pd.DataFrame(test_rows).T.sort_values('Accuracy', ascending=False)
print("Held-out TEST performance (for reporting only, not used for selection):")
display(performance_df.round(4))

best_model = fitted[best_model_name]
print(f"\nSelected model: {best_model_name}")

In [ ]:
# Confusion matrix of the selected model
pred_best = best_model.predict(X_test)
plt.figure(figsize=(12, 10))
sns.heatmap(confusion_matrix(y_test_encoded, pred_best), annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()

## 6. Export the Best Model as a Pickle
The bundle contains the model, label encoder, feature order and metrics, so it can be loaded and used without retraining.

In [ ]:
PICKLE_PATH = "crop_recommendation_best_model.pkl"

bundle = {
    'model': best_model,
    'model_name': best_model_name,
    'label_encoder': label_encoder,
    'feature_names': FEATURES,
    'crop_profile': crop_profile,
    'cv_results': cv_df,
    'test_metrics': performance_df.loc[best_model_name].to_dict(),
    'sklearn_version': sklearn.__version__,
}
with open(PICKLE_PATH, 'wb') as f:
    pickle.dump(bundle, f)
print(f"Saved: {PICKLE_PATH}  ({os.path.getsize(PICKLE_PATH)/1024:.0f} KB)")

# Verify: reload and check predictions are identical
with open(PICKLE_PATH, 'rb') as f:
    loaded = pickle.load(f)
same = (loaded['model'].predict(X_test) == best_model.predict(X_test)).all()
print("Reloaded model gives identical predictions:", bool(same))

# Download in Colab (ignored elsewhere)
try:
    from google.colab import files
    files.download(PICKLE_PATH)
except ImportError:
    pass

**Using the pickle later** (use the same scikit-learn version it was saved with):
```python
import pickle, pandas as pd
b = pickle.load(open("crop_recommendation_best_model.pkl", "rb"))
row = pd.DataFrame([{'N': 80, 'P': 40, 'K': 40, 'temperature': 22.0,
                     'humidity': 85.0, 'ph': 6.5, 'rainfall': 200.0}])[b['feature_names']]
probs = b['model'].predict_proba(row)[0]
print(pd.Series(probs, index=b['label_encoder'].classes_).nlargest(5))
```

## 7. Feature Importance with SHAP (selected model)

In [ ]:
import shap

tree_types = [RandomForestClassifier]
if lgb is not None: tree_types.append(lgb.LGBMClassifier)
if xgb is not None: tree_types.append(xgb.XGBClassifier)

def mean_abs_shap(sv, n_features):
    """Mean |SHAP| per feature, averaged over samples AND all classes."""
    if isinstance(sv, list):                       # list of (n, f), one per class
        return np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
    sv = np.abs(np.asarray(sv))
    if sv.ndim == 2:
        return sv.mean(axis=0)
    if sv.shape[1] == n_features:                  # (n, f, c)
        return sv.mean(axis=(0, 2))
    if sv.shape[2] == n_features:                  # (n, c, f)
        return sv.mean(axis=(0, 1))
    return sv.mean(axis=(1, 2))                    # (f, n, c)

if isinstance(best_model, tuple(tree_types)):
    explainer = shap.TreeExplainer(best_model)
    X_shap = X_test.values
    shap_values = explainer.shap_values(X_shap)
else:
    print(f"Using KernelExplainer for {best_model_name} (slower, sampled)...")
    background = X_train.sample(50, random_state=SEED).values
    X_shap = X_test.sample(30, random_state=SEED).values
    predict_fn = lambda a: best_model.predict_proba(pd.DataFrame(a, columns=FEATURES))
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(X_shap, nsamples=100)

importance = pd.Series(mean_abs_shap(shap_values, len(FEATURES)), index=FEATURES).sort_values()
plt.figure(figsize=(8, 5))
importance.plot.barh(color='steelblue')
plt.xlabel('Mean |SHAP value| (averaged over all crop classes)')
plt.title(f'Global Feature Importance - {best_model_name}')
plt.tight_layout(); plt.show()
display(importance.sort_values(ascending=False).round(4))

## 7b. New Input to Explain and Predict

In [ ]:
new_input_df = pd.DataFrame({'N': [80], 'P': [40], 'K': [40], 'temperature': [22.0],
                             'humidity': [85.0], 'ph': [6.5], 'rainfall': [200.0]})[FEATURES]
display(new_input_df)

## 8. Counterfactual Explanations with DiCE (selected model)

In [ ]:
import dice_ml

class DynamicClassifierWrapper:
    def __init__(self, model, label_encoder):
        self.model = model
        self.classes_ = label_encoder.classes_
        self.n_classes_ = len(label_encoder.classes_)
    def predict(self, X):        return self.model.predict(X)
    def predict_proba(self, X):  return self.model.predict_proba(X)

dice_data = dice_ml.Data(
    dataframe=pd.concat([X_train, pd.Series(y_train_encoded, name='label', index=X_train.index)], axis=1),
    continuous_features=FEATURES,
    outcome_name='label',
    outcome_values=list(range(len(label_encoder.classes_))))

dice_model = dice_ml.Model(model=DynamicClassifierWrapper(best_model, label_encoder),
                           backend='sklearn', model_type='classifier')
dice = dice_ml.Dice(dice_data, dice_model, method='random')
print(f"DiCE ready for: {best_model_name}")

In [ ]:
query_instance = new_input_df.copy()

orig_enc = best_model.predict(query_instance)[0]
orig_label = label_encoder.inverse_transform([orig_enc])[0]
print("Input:"); display(query_instance)
print(f"Current prediction ({best_model_name}): {orig_label}\n")

rng = random.Random(SEED)                       # reproducible choice of target crops
alternatives = [c for c in range(len(label_encoder.classes_)) if c != orig_enc]
targets = rng.sample(alternatives, 3)

records = []
for t in targets:
    t_label = label_encoder.inverse_transform([t])[0]
    try:
        cf = dice.generate_counterfactuals(query_instance, total_CFs=1,
                                           desired_class=t, verbose=False)
        cf_df = cf.cf_examples_list[0].final_cfs_df
        if cf_df is None or len(cf_df) == 0:
            print(f"No counterfactual found for {t_label}"); continue
        cf_feat = cf_df[FEATURES].iloc[[0]]
        cf_pred = label_encoder.inverse_transform(best_model.predict(cf_feat))[0]
        rec = {'Original_Crop': orig_label, 'Target_Crop': t_label, 'Predicted_CF_Crop': cf_pred}
        for f in FEATURES:
            rec[f'Change_{f}'] = float(cf_feat[f].iloc[0] - query_instance[f].iloc[0])
        records.append(rec)
    except Exception as e:
        print(f"Error for {t_label}: {e}")

cf_summary = pd.DataFrame(records)
print("Changes needed in ALL features (0 = unchanged):")
display(cf_summary.round(3))

In [ ]:
if not cf_summary.empty:
    fig, axes = plt.subplots(4, 2, figsize=(13, 14))
    for ax, f in zip(axes.ravel(), FEATURES):
        sns.barplot(data=cf_summary, x='Target_Crop', y=f'Change_{f}',
                    hue='Target_Crop', legend=False, palette='viridis', ax=ax)
        ax.axhline(0, color='grey', ls='--', lw=0.8)
        ax.set_title(f'Change in {f}'); ax.set_xlabel('')
    axes.ravel()[-1].axis('off')
    plt.tight_layout(); plt.show()

## 9. Top-5 Crop Predictions for the New Input

In [ ]:
probs = best_model.predict_proba(new_input_df)[0]
top_5_crops = (pd.DataFrame({'Crop': label_encoder.classes_, 'Probability': probs})
               .sort_values('Probability', ascending=False).head(5).reset_index(drop=True))
print(f"Top 5 crops using {best_model_name}:")
display(top_5_crops.round(4))

## 10. Soil Analysis for the Recommended Crop
Your **soil** values (N, P, K, pH) are compared with the ideal range for the recommended crop. Each parameter is marked **Low**, **Optimal** or **High**, with the adjustment needed to reach the range.

Notes: ranges come from the dataset (10th to 90th percentile per crop), and N, P, K are relative to the dataset's own scale. The suggestions are general guidance, so confirm with a local soil-testing lab or agronomist before applying fertilizer or amendments.

In [ ]:
SOIL = ['N', 'P', 'K', 'ph']
SOIL_NAMES = {'N': 'Nitrogen (N)', 'P': 'Phosphorus (P)', 'K': 'Potassium (K)', 'ph': 'Soil pH'}

SUGGESTIONS = {
    ('N', 'Low'):   'Add nitrogen: urea or ammonium sulphate, compost/FYM, or rotate with a legume crop.',
    ('N', 'High'):  'Reduce nitrogen fertilizer; excess N can cause lush weak growth and leaching.',
    ('P', 'Low'):   'Add phosphorus: SSP or DAP, bone meal, or well-rotted compost.',
    ('P', 'High'):  'Skip phosphorus fertilizer this season; excess P can lock up zinc and iron.',
    ('K', 'Low'):   'Add potassium: muriate of potash (MOP), sulphate of potash, or wood ash.',
    ('K', 'High'):  'Skip potassium fertilizer this season; excess K can limit magnesium and calcium uptake.',
    ('ph', 'Low'):  'Soil is too acidic for this crop: apply agricultural lime or dolomite to raise pH.',
    ('ph', 'High'): 'Soil is too alkaline for this crop: use elemental sulphur, gypsum or organic matter to lower pH.',
}

def ph_class(v):
    if v < 5.5: return 'Strongly acidic'
    if v < 6.5: return 'Slightly acidic'
    if v <= 7.5: return 'Neutral'
    if v <= 8.5: return 'Alkaline'
    return 'Strongly alkaline'

def analyse_soil(sample, crop):
    prof = crop_profile[crop]
    rows = []
    for f in SOIL:
        v = float(sample[f].iloc[0])
        lo, med, hi = prof.loc['low', f], prof.loc['median', f], prof.loc['high', f]
        if v < lo:
            status, change = 'Low', lo - v
        elif v > hi:
            status, change = 'High', hi - v
        else:
            status, change = 'Optimal', 0.0
        rows.append({
            'Parameter': SOIL_NAMES[f],
            'Your value': round(v, 2),
            'Ideal range': f'{lo:.1f} - {hi:.1f}',
            'Typical': round(med, 1),
            'Status': status,
            'Change to reach range': round(change, 2),
            'Suggestion': SUGGESTIONS.get((f, status), 'No action needed.'),
        })
    return pd.DataFrame(rows)

def soil_suitability(sample, crops):
    out = []
    for crop in crops:
        rep = analyse_soil(sample, crop)
        bad = rep.loc[rep['Status'] != 'Optimal', 'Parameter'].tolist()
        out.append({'Crop': crop,
                    'Soil match (%)': 100 * (len(SOIL) - len(bad)) / len(SOIL),
                    'Out of range': ', '.join(bad) if bad else 'None'})
    return pd.DataFrame(out)

top_crop = top_5_crops.loc[0, 'Crop']
ph_value = float(new_input_df['ph'].iloc[0])
print('Soil pH class:', ph_class(ph_value), '(general guide)')
print()

print('Soil suitability for the top-5 crops:')
display(soil_suitability(new_input_df, top_5_crops['Crop']).round(1))

print()
print('Detailed soil analysis for the top recommendation:', top_crop)
soil_report = analyse_soil(new_input_df, top_crop)
display(soil_report)

In [ ]:
# Climate check for the top crop (cannot be changed like soil, but shows risk)
prof = crop_profile[top_crop]
climate_rows = []
for f in ['temperature', 'humidity', 'rainfall']:
    v = float(new_input_df[f].iloc[0])
    lo, hi = prof.loc['low', f], prof.loc['high', f]
    status = 'Low' if v < lo else 'High' if v > hi else 'Optimal'
    climate_rows.append({'Parameter': f, 'Your value': round(v, 2),
                         'Ideal range': f'{lo:.1f} - {hi:.1f}', 'Status': status})
print(f'Climate check for {top_crop}:')
display(pd.DataFrame(climate_rows))
print('Climate cannot be adjusted like soil; if it is out of range, consider irrigation, '
      'shade/mulching, or a different sowing season.')

In [ ]:
# Visual: where each soil value sits against the ideal range for the top crop
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, f in zip(axes, SOIL):
    lo, med, hi = (crop_profile[top_crop].loc[k, f] for k in ['low', 'median', 'high'])
    v = float(new_input_df[f].iloc[0])
    ok = lo <= v <= hi
    ax.barh([0], [hi - lo], left=lo, color='#9ad0a0', height=0.4, label='Ideal range')
    ax.axvline(med, color='green', ls='--', lw=1, label='Typical')
    ax.scatter([v], [0], s=140, color='#2a7' if ok else '#d33', zorder=3, label='Your value')
    pad = (hi - lo) * 0.6 + 1e-6
    ax.set_xlim(min(lo, v) - pad, max(hi, v) + pad)
    ax.set_yticks([]); ax.set_title(SOIL_NAMES[f], fontsize=10)
axes[0].legend(fontsize=7, loc='upper left')
plt.suptitle(f'Soil values vs ideal range for {top_crop}', y=1.05)
plt.tight_layout(); plt.show()

## 11. Economic Analysis of the Top-5 Crops
Units: yield is in kg/ha, price and cost are in Rs per **quintal** (1 quintal = 100 kg).
- Yield (quintals/ha) = yield_kg_per_ha / 100
- Revenue (Rs/ha) = yield_quintals x price
- Cost (Rs/ha) = yield_quintals x cost per quintal
- Profit = Revenue - Cost; ROI = Profit / Cost x 100

In [ ]:
eco_df = load_csv("econmic.csv")
eco_df.columns = eco_df.columns.str.strip().str.lower()
eco_df['crop'] = eco_df['crop'].str.strip().str.lower()

merged = top_5_crops.assign(crop=top_5_crops['Crop'].str.strip().str.lower()).merge(
    eco_df[['crop', 'yield_kg_per_ha', 'price_rs_per_quintal', 'cost_rs_per_quintal']],
    on='crop', how='left')

missing = merged[merged['yield_kg_per_ha'].isna()]['Crop'].tolist()
if missing:
    print("WARNING - no economic data for:", missing)

merged['Yield (q/ha)'] = merged['yield_kg_per_ha'] / 100.0
merged['Revenue (Rs/ha)'] = merged['Yield (q/ha)'] * merged['price_rs_per_quintal']
merged['Cost (Rs/ha)'] = merged['Yield (q/ha)'] * merged['cost_rs_per_quintal']
merged['Profit (Rs/ha)'] = merged['Revenue (Rs/ha)'] - merged['Cost (Rs/ha)']
merged['ROI (%)'] = merged['Profit (Rs/ha)'] / merged['Cost (Rs/ha)'] * 100.0

merged['ML Rank'] = merged['Probability'].rank(ascending=False, method='first').astype(int)
merged['Profit Rank'] = merged['Profit (Rs/ha)'].rank(ascending=False, method='first')
merged['ROI Rank'] = merged['ROI (%)'].rank(ascending=False, method='first')

economic_table = merged[['Crop', 'Probability', 'ML Rank', 'Revenue (Rs/ha)', 'Cost (Rs/ha)',
                         'Profit (Rs/ha)', 'Profit Rank', 'ROI (%)', 'ROI Rank']].sort_values('ML Rank')
print("ML ranking vs economic ranking (top-5 crops):")
display(economic_table.round(2))